# Step 2: Generate Image Descriptions with GPT-4o Vision

Send extracted images to Azure OpenAI GPT-4o to get text descriptions. These descriptions will be merged into each page's text so images become searchable in the RAG pipeline.

In [28]:
%pip install openai azure-identity python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import base64
import time
from pathlib import Path
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv

load_dotenv(override=True)

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

client = OpenAI(
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=token_provider,
)
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

##print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Deployment: {CHAT_DEPLOYMENT}")

# Load parsed pages from Step 1
with open("extracted_data/parsed_pages.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

# Find pages that have images
pages_with_images = [p for p in pages if p["has_images"]]
total_images = sum(len(p["images"]) for p in pages_with_images)

print(f"\nLoaded {len(pages)} pages")
print(f"Pages with images: {len(pages_with_images)}")
print(f"Total images to describe: {total_images}")


Deployment: gpt-4.1

Loaded 3623 pages
Pages with images: 217
Total images to describe: 218


In [30]:
def describe_image(image_path: str, page_text: str) -> str:
    """Send an image to GPT-4o Vision and get a text description."""
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode("utf-8")
    
    # Include surrounding page text for context
    context_snippet = page_text[:500] if page_text else "No surrounding text."
    
    response = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {
                "role": "system",
                "content": "You describe images from a book. Give a concise but detailed description of what the image shows. If it contains text, include that text. If it's a diagram, describe its structure."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"Describe this image. Context from the page: {context_snippet}"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_b64}",
                            "detail": "low"
                        }
                    }
                ]
            }
        ],
        max_tokens=300,
    )
    return response.choices[0].message.content

# Quick test with first image
test_page = pages_with_images[0]
test_img = test_page["images"][0]
test_path = os.path.join("extracted_data", "images", test_img["filename"])

print(f"Testing with: {test_img['filename']} (page {test_page['page_number']})")
desc = describe_image(test_path, test_page["text"])
print(f"\nDescription:\n{desc}")

Testing with: page_1_img_1.jpeg (page 1)

Description:
The image shows the cover of a book titled "**Harry Potter: The Complete Collection**" by **J.K. Rowling**. The background is a dark reddish-brown color with hints of swirling patterns. At the center, there are round eyeglasses with a lightning bolt scar above them, both symbols strongly associated with the character Harry Potter. The book title is prominently displayed in large, stylized white and gold fonts, and the author's name is in smaller white text at the bottom. The design is simple and iconic, immediately recognizable as being related to the Harry Potter series.


In [31]:
# Process all images — with progress tracking and retry on rate limits
PROGRESS_FILE = "extracted_data/image_descriptions_progress.json"

# Resume from progress if exists (in case of interruption)
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
        descriptions = json.load(f)
    print(f"Resuming — {len(descriptions)} images already described")
else:
    descriptions = {}

processed = 0
failed = 0

for page in pages_with_images:
    for img in page["images"]:
        key = img["filename"]
        if key in descriptions:
            continue  # already done
        
        image_path = os.path.join("extracted_data", "images", img["filename"])
        try:
            desc = describe_image(image_path, page["text"])
            descriptions[key] = {
                "page_number": page["page_number"],
                "filename": img["filename"],
                "description": desc
            }
            processed += 1
            
            if processed % 10 == 0:
                # Save progress every 10 images
                with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
                    json.dump(descriptions, f, ensure_ascii=False, indent=2)
                print(f"  Progress: {processed + len(descriptions) - processed}/{total_images}")
                
        except Exception as e:
            if "429" in str(e):  # Rate limited
                print(f"  Rate limited — waiting 30s...")
                time.sleep(30)
                # Retry once
                try:
                    desc = describe_image(image_path, page["text"])
                    descriptions[key] = {
                        "page_number": page["page_number"],
                        "filename": img["filename"],
                        "description": desc
                    }
                    processed += 1
                except Exception:
                    failed += 1
                    print(f"  Failed: {img['filename']}")
            else:
                failed += 1
                print(f"  Error on {img['filename']}: {e}")

# Final save
with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
    json.dump(descriptions, f, ensure_ascii=False, indent=2)

print(f"\n=== Done ===")
print(f"Described: {len(descriptions)}/{total_images}")
print(f"Failed: {failed}")

  Progress: 10/218
  Progress: 20/218
  Progress: 30/218
  Progress: 40/218
  Progress: 50/218
  Progress: 60/218
  Error on page_898_img_1.jpeg: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'indirect_attack': {'detected': False, 'filtered': False}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': True, 'severity': 'medium'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  Progress: 70/218
  Progress: 80/218
  Progress: 90/218
  Progress: 

In [33]:
# Merge image descriptions back into pages data
desc_lookup = {d["filename"]: d["description"] for d in descriptions.values()}

for page in pages:
    image_descs = []
    for img in page.get("images", []):
        desc = desc_lookup.get(img["filename"], "")
        if desc:
            image_descs.append(desc)
            img["description"] = desc
    
    # Append image descriptions to page text
    if image_descs:
        page["text_with_images"] = page["text"] + "\n\n[Image descriptions]\n" + "\n".join(image_descs)
    else:
        page["text_with_images"] = page["text"]

# Save enriched pages
output_path = "extracted_data/parsed_pages_with_images.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

print(f"Saved enriched data: {output_path}")
print(f"Pages with image descriptions: {sum(1 for p in pages if p.get('text_with_images') != p['text'])}")
print(f"\nDone! Ready for Step 3: Chunking & Embedding.")

Saved enriched data: extracted_data/parsed_pages_with_images.json
Pages with image descriptions: 216

Done! Ready for Step 3: Chunking & Embedding.
